# Data Preparation for Survival Analysis
This notebook processes the raw transactions dataset, containing over 20 million rows with multiple transactions per user. The goal is to construct a clean survival dataset with exactly one row per user.

### Key Points
- **Churn Definition**: Churn occurs if a user fails to renew within 0 days of their `membership_expire_date`. For this analysis, we use the first time this happens (the first observed churn).
- **Right Censoring**: If the observation window (ending 2017-03-31) concludes within 30 days of a user's final `membership_expire_date`, they are right-censored. Because it is impossible to observe whether they would have renewed within the full 30-day window, their eventual churn date is unknown, hence the censoring.
- **Backdated Records**: Transactions with negative subscription durations (where the expiry date is before the transaction date) are excluded.
- **Overlapping Subscriptions**: The chronological sequence of transactions is used. Since a new transaction may extend an existing expiry, we track the sequence carefully and use the first relevant transaction.
- **Endpoints**: The survival endpoint is anchored to the user's first observed churn event. If they never churned, their final censored event is used.
- **Baseline Covariates**: To avoid data leakage, subscription covariates (like payment methods, auto-renew status, etc.) are extracted strictly from the user's first transaction.


In [1]:
# Import libraries
import gc
import os

import numpy as np
import pandas as pd

## Load Data

In [4]:
# Load datasets with optimised datatypes to save memory
dtypes = {
    'payment_method_id': 'int8',
    'payment_plan_days': 'int16',
    'plan_list_price': 'int16',
    'actual_amount_paid': 'int16',
    'is_auto_renew': 'int8',
    'is_cancel': 'int8'
}

transactions = pd.read_csv('./data/transactions.csv', dtype=dtypes)
trans2 = pd.read_csv('./data/transactions_v2.csv', dtype=dtypes)

# Combine and delete trans2 to free memory
transactions = pd.concat([transactions, trans2], ignore_index=True)
del trans2
gc.collect()

# Parse dates
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], format='%Y%m%d')
transactions['membership_expire_date'] = pd.to_datetime(transactions['membership_expire_date'], format='%Y%m%d')

# Drop duplicates
initial_len = len(transactions)
transactions.drop_duplicates(inplace=True)
print(f'Dropped {initial_len - len(transactions):,} exact duplicates.')

print(f'Loaded {len(transactions):,} unique transactions')
print(f'Unique users: {transactions["msno"].nunique():,}')

Dropped 3,339 exact duplicates.
Loaded 22,975,416 unique transactions
Unique users: 2,426,143


In [6]:
transactions.shape

(22975416, 9)

In [7]:
transactions.columns

Index(['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price',
       'actual_amount_paid', 'is_auto_renew', 'transaction_date',
       'membership_expire_date', 'is_cancel'],
      dtype='object')

In [5]:
transactions.head()

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=,41,30,129,129,1,2015-09-30,2015-11-01,0
1,AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=,41,30,149,149,1,2015-09-30,2015-10-31,0
2,UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=,41,30,129,129,1,2015-09-30,2016-04-27,0
3,M1C56ijxozNaGD0t2h68PnH2xtx5iO5iR2MVYQB6nBI=,39,30,149,149,1,2015-09-30,2015-11-28,0
4,yvj6zyBUaqdbUQSrKsrZ+xNDVM62knauSZJzakS9OW4=,39,30,149,149,1,2015-09-30,2015-11-21,0


In [8]:
transactions.isna().sum()

msno                      0
payment_method_id         0
payment_plan_days         0
plan_list_price           0
actual_amount_paid        0
is_auto_renew             0
transaction_date          0
membership_expire_date    0
is_cancel                 0
dtype: int64

## Clean Data
Remove negative subscription lengths (backdated records).

In [9]:
# Derive subscription length
transactions['subscription_length_days'] = (
    transactions['membership_expire_date'] - transactions['transaction_date']
).dt.days

# Keep only valid forward-looking transactions
transactions.drop(transactions[transactions['subscription_length_days'] < 0].index, inplace=True)

# Sort chronologically in-place
transactions.sort_values(["msno", "transaction_date"], inplace=True)

transactions_clean = transactions
print(f'Records after cleaning backdated transactions: {len(transactions_clean):,}')

Records after cleaning backdated transactions: 22,816,913


## Define Churn and Censoring Logic

In [10]:
# Calculate gap to next transaction
transactions_clean['next_transaction_date'] = (
    transactions_clean.groupby('msno')['transaction_date'].shift(-1)
)

transactions_clean['gap_after_expiry'] = (
    transactions_clean['next_transaction_date'] - transactions_clean['membership_expire_date']
).dt.days

OBSERVATION_END = transactions_clean['transaction_date'].max()
CHURN_WINDOW = pd.Timedelta(days=30)

# Tag final transaction per user
transactions_clean['is_last_transaction'] = (
    transactions_clean.groupby('msno')['transaction_date'].transform('max') == transactions_clean['transaction_date']
)

# Observed churn: Next transaction exists, but gap > 30 days
observed_churn = (
    transactions_clean['next_transaction_date'].notna() & 
    (transactions_clean['gap_after_expiry'] > 30)
)

# Final transaction churn: It's the final transaction AND 30 days have passed since expiry
final_transaction_observed_long_enough = (
    transactions_clean['is_last_transaction'] & 
    transactions_clean['membership_expire_date'].notna() & 
    (transactions_clean['membership_expire_date'] + CHURN_WINDOW <= OBSERVATION_END)
)
final_transaction_churn = (
    final_transaction_observed_long_enough & 
    transactions_clean['next_transaction_date'].isna()
)

# Combine into a single churn flag
transactions_clean['is_churn_point'] = observed_churn | final_transaction_churn

# Censored: User reaches observation end before we can establish churn
transactions_clean['is_censored'] = (
    transactions_clean['is_last_transaction'] & ~transactions_clean['is_churn_point']
)

In [11]:
transactions_clean.head()

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel,subscription_length_days,next_transaction_date,gap_after_expiry,is_last_transaction,is_churn_point,is_censored
6797850,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,35,7,0,0,0,2016-09-09,2016-09-14,0,5,NaT,NaN,True,True,False
1521480,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,38,410,1788,1788,0,2015-11-21,2017-01-04,0,410,2016-10-23,-73.0,False,False,False
21797460,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,22,395,1599,1599,0,2016-10-23,2018-02-06,0,471,NaT,NaN,True,False,True
1498592,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,41,30,99,99,1,2016-11-16,2016-12-15,0,29,2016-12-15,0.0,False,False,False
17923235,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,41,30,99,99,1,2016-12-15,2017-01-15,0,31,2017-01-15,0.0,False,False,False


## Extract Endpoints and Baseline Covariates

In [13]:
# Get the firt churn event for users who churned
churned_endpoints = (
    transactions_clean[transactions_clean['is_churn_point']]
    .groupby('msno', as_index=False)
    .first()
)

# Get the censored endpoint for users who never churned
censored_users_mask = transactions_clean['is_censored'] & ~transactions_clean['msno'].isin(churned_endpoints['msno'])
censored_endpoints = (
    transactions_clean[censored_users_mask]
    .groupby('msno', as_index=False)
    .last()
)

# Combine endpoints
final_endpoints = pd.concat([churned_endpoints, censored_endpoints], ignore_index=True)
print(f'Total endpoint records: {len(final_endpoints):,}')

Total endpoint records: 2,417,562


In [14]:
# Extract the first transaction per user for baseline covariates
baseline_covariates = (
    transactions_clean
    .groupby('msno', as_index=False)
    .first()
)[['msno', 'transaction_date', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'is_cancel']]

# Rename transaction_date to start_date for clarity
baseline_covariates = baseline_covariates.rename(columns={'transaction_date': 'start_date'})

## Build Survival Table

In [15]:
survival_table = final_endpoints[['msno', 'membership_expire_date', 'is_churn_point']].copy()

# 1 = observed churn, 0 = right-censored
survival_table['event'] = survival_table['is_churn_point'].astype(int)

# Set endpoint date
survival_table['endpoint_date'] = np.where(
    survival_table['event'] == 1,
    survival_table['membership_expire_date'],
    OBSERVATION_END
)
survival_table['endpoint_date'] = pd.to_datetime(survival_table['endpoint_date'])

# Merge with baseline covariates
survival_table = survival_table.merge(baseline_covariates, on='msno', how='left')

# Calculate total duration in days
survival_table['duration_days'] = (
    survival_table['endpoint_date'] - survival_table['start_date']
).dt.days

# Keep final columns
final_cols = [
    'msno', 'start_date', 'endpoint_date', 'duration_days', 'event',
    'payment_method_id', 'payment_plan_days', 'plan_list_price', 
    'actual_amount_paid', 'is_auto_renew', 'is_cancel'
]
survival_table = survival_table[final_cols]

print(f'Survival table shape: {survival_table.shape}')

Survival table shape: (2417562, 11)


In [16]:
survival_table.head()

,msno,start_date,endpoint_date,duration_days,event,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,is_cancel
0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,2016-09-09,2016-09-14,5,1,35,7,0,0,0,0
1,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2015-01-31,2016-03-19,413,1,39,31,149,149,1,0
2,++/Gw1B9K+XOlB3hLTloeUK2QlCa2m+BJ8TrzGf7djI=,2015-01-13,2016-03-01,413,1,40,31,149,149,1,0
3,++/TR7WI15q2ZCtOXmoap7jR+kEhbMVE5swOqsfqpqI=,2015-01-24,2015-09-25,244,1,11,31,149,149,1,0
4,++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50=,2015-01-31,2016-03-23,417,1,39,31,149,149,1,0


## Checks

In [ ]:
# Assert exactly one row per user
assert len(survival_table) == survival_table['msno'].nunique(), 'ERROR: Multiple rows per user detected!'

# Assert no negative durations
assert (survival_table['duration_days'] < 0).sum() == 0, 'ERROR: Negative durations detected!'

# Print Event Breakdown
print('Event Breakdown (1=Churn, 0=Censored):')
print(survival_table['event'].value_counts(normalize=True).round(3) * 100)

# Check for missing values
print('\nMissing Values:')
print(survival_table.isna().sum())

Event Breakdown (1=Churn, 0=Censored):
event
1    62.5
0    37.5
Name: proportion, dtype: float64

Missing Values:
msno                  0
start_date            0
endpoint_date         0
duration_days         0
event                 0
payment_method_id     0
payment_plan_days     0
plan_list_price       0
actual_amount_paid    0
is_auto_renew         0
is_cancel             0
dtype: int64


## Export

In [18]:
os.makedirs('../data', exist_ok=True)
survival_table.to_csv('../data/survival_table.csv', index=False)
print('Saved survival_table.csv')

Saved survival_table.csv
